# 📘 Model Responses: Performance, Cost & Resource Analysis

**Deep dive into 5 VLM inference outputs** - latency, cost, tokens, GPU usage, and response characteristics.

## Sections
1. Introduction & Setup
2. Model Performance Overview
3. Latency Analysis ⏱️
4. Cost Analysis 💰
5. Token Usage Analysis 🔢
6. GPU Metrics Analysis 🖥️
7. Response Quality & Characteristics ✅
8. Confidence Score Analysis 🎯
9. Performance by Task Type 📊
10. Performance by Ground Truth Type 🎲
11. Performance by Data Split 🔀
12. Cross-Model Correlation 🔗
13. Cost-Performance Tradeoffs ⚖️
14. Final Summary & Recommendations 📝

---
## 1. Introduction & Setup

In [1]:
# === Path Setup ===
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent
ROOT_DIR = ARTEMIS_DIR.parent

for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 ARTEMIS_DIR: {ARTEMIS_DIR}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
from PIL import Image
import io
import warnings
warnings.filterwarnings('ignore')

from ares.db.connection import get_engine
from ares.configs.db_config import TABLES, MODEL_NAMES

engine = get_engine()

# Publication-quality styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Model color palette
MODEL_COLORS = {
    'deepseek_ocr': '#FF6B6B',
    'qwen2_5_vl_3b': '#4ECDC4',
    'qwen2_5_vl_7b': '#45B7D1',
    'qwen3_vl_8b_thinking': '#96CEB4',
    'gemma_3_27b': '#FFEAA7'
}

print('✅ Connected!')

📁 ARTEMIS_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final
✅ Connected!


In [4]:
# Load all data
df_samples = pd.read_sql('SELECT * FROM vlm_samples', engine)
df_responses = pd.read_sql('SELECT * FROM vlm_responses', engine)
df_evaluations = pd.read_sql('SELECT * FROM vlm_evaluations', engine)

# Merge for comprehensive analysis
df = df_responses.merge(df_samples[['sample_id', 'source_config', 'router_task', 'ground_truth_type', 'split']], 
                        on='sample_id', how='left')
df = df.merge(df_evaluations[['response_id', 'glider_score', 'vlm_judge_score']], 
              on='response_id', how='left')

print(f"📊 Loaded {len(df_samples):,} samples, {len(df_responses):,} responses, {len(df_evaluations):,} evaluations")
print(f"📊 Merged dataset: {len(df):,} rows")

KeyError: "['split'] not in index"

---
## 2. Model Performance Overview

Summary table with all key metrics per model.

In [ ]:
# Comprehensive summary table
summary_data = []
for model in MODEL_NAMES:
    model_df = df[df['model_name'] == model]
    ok_df = model_df[model_df['ok'] == True]
    
    summary_data.append({
        'Model': model,
        'Total Samples': len(model_df),
        'Success Rate (%)': round(len(ok_df) / len(model_df) * 100, 1) if len(model_df) > 0 else 0,
        'Avg Latency (ms)': round(model_df['latency_ms'].mean(), 0),
        'P95 Latency (ms)': round(model_df['latency_ms'].quantile(0.95), 0),
        'Total Cost ($)': round(model_df['estimated_cost'].sum(), 4),
        'Cost/Sample ($)': round(model_df['estimated_cost'].mean(), 6),
        'Avg Input Tokens': round(model_df['input_tokens'].mean(), 0),
        'Avg Output Tokens': round(model_df['output_tokens'].mean(), 0),
        'Avg Confidence': round(model_df['confidence_score'].mean(), 3),
        'Avg Glider Score': round(model_df['glider_score'].mean(), 3) if 'glider_score' in model_df.columns else np.nan,
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary)

In [ ]:
# Leaderboard visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics = ['Success Rate (%)', 'Avg Latency (ms)', 'Cost/Sample ($)', 
           'Avg Output Tokens', 'Avg Confidence', 'Avg Glider Score']
colors = [MODEL_COLORS.get(m, '#888888') for m in df_summary['Model']]

for ax, metric in zip(axes.flatten(), metrics):
    sorted_df = df_summary.sort_values(metric, ascending=(metric != 'Avg Latency (ms)'))
    ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel(metric)

plt.suptitle('Model Performance Leaderboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Latency Analysis ⏱️

In [ ]:
# Latency distribution histograms
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, model in zip(axes.flatten()[:5], MODEL_NAMES):
    model_data = df[df['model_name'] == model]['latency_ms'].dropna()
    ax.hist(model_data, bins=50, color=MODEL_COLORS[model], alpha=0.7, edgecolor='white')
    ax.axvline(model_data.median(), color='red', linestyle='--', label=f'Median: {model_data.median():.0f}ms')
    ax.axvline(model_data.quantile(0.95), color='orange', linestyle=':', label=f'P95: {model_data.quantile(0.95):.0f}ms')
    ax.set_title(model, fontweight='bold')
    ax.set_xlabel('Latency (ms)')
    ax.legend(fontsize=8)

axes.flatten()[5].axis('off')
plt.suptitle('Latency Distribution by Model', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots and violin plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette = [MODEL_COLORS[m] for m in MODEL_NAMES]
sns.boxplot(data=df, x='model_name', y='latency_ms', palette=palette, ax=axes[0], order=MODEL_NAMES)
axes[0].set_title('Latency Box Plot', fontweight='bold')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Latency (ms)')
axes[0].tick_params(axis='x', rotation=45)

sns.violinplot(data=df, x='model_name', y='latency_ms', palette=palette, ax=axes[1], order=MODEL_NAMES)
axes[1].set_title('Latency Violin Plot', fontweight='bold')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Latency (ms)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Task × Model latency
latency_pivot = df.pivot_table(values='latency_ms', index='router_task', columns='model_name', aggfunc='mean')

plt.figure(figsize=(14, 12))
sns.heatmap(latency_pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Mean Latency (ms) by Task × Model', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Task')
plt.tight_layout()
plt.show()

---
## 4. Cost Analysis 💰

In [ ]:
# Total cost breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cost_by_model = df.groupby('model_name')['estimated_cost'].sum().sort_values(ascending=False)
colors = [MODEL_COLORS.get(m, '#888888') for m in cost_by_model.index]

axes[0].pie(cost_by_model, labels=cost_by_model.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Total Cost Distribution', fontweight='bold')

# Cost per correct answer
cost_efficiency = []
for model in MODEL_NAMES:
    model_df = df[df['model_name'] == model]
    correct = len(model_df[model_df['glider_score'] >= 0.5]) if 'glider_score' in model_df.columns else len(model_df[model_df['ok'] == True])
    total_cost = model_df['estimated_cost'].sum()
    cost_efficiency.append({'Model': model, 'Cost/Correct': total_cost / max(correct, 1) * 1000})  # per 1000 correct

df_cost_eff = pd.DataFrame(cost_efficiency).sort_values('Cost/Correct')
colors = [MODEL_COLORS.get(m, '#888888') for m in df_cost_eff['Model']]
axes[1].barh(df_cost_eff['Model'], df_cost_eff['Cost/Correct'], color=colors)
axes[1].set_title('Cost per 1000 Correct Answers ($)', fontweight='bold')
axes[1].set_xlabel('Cost ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Cumulative cost curves
plt.figure(figsize=(12, 6))

for model in MODEL_NAMES:
    model_costs = df[df['model_name'] == model]['estimated_cost'].sort_index().cumsum()
    plt.plot(range(len(model_costs)), model_costs, label=model, color=MODEL_COLORS[model], linewidth=2)

plt.xlabel('Sample Number')
plt.ylabel('Cumulative Cost ($)')
plt.title('Cumulative Cost Over Samples', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Token Usage Analysis 🔢

In [ ]:
# Token distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Input tokens
for model in MODEL_NAMES:
    data = df[df['model_name'] == model]['input_tokens'].dropna()
    axes[0].hist(data, bins=50, alpha=0.5, label=model, color=MODEL_COLORS[model])
axes[0].set_title('Input Token Distribution', fontweight='bold')
axes[0].set_xlabel('Input Tokens')
axes[0].legend(fontsize=8)

# Output tokens
for model in MODEL_NAMES:
    data = df[df['model_name'] == model]['output_tokens'].dropna()
    axes[1].hist(data, bins=50, alpha=0.5, label=model, color=MODEL_COLORS[model])
axes[1].set_title('Output Token Distribution', fontweight='bold')
axes[1].set_xlabel('Output Tokens')
axes[1].legend(fontsize=8)

# Verbosity ratio
df['verbosity'] = df['output_tokens'] / df['input_tokens'].replace(0, np.nan)
verbosity = df.groupby('model_name')['verbosity'].mean().reindex(MODEL_NAMES)
colors = [MODEL_COLORS[m] for m in MODEL_NAMES]
axes[2].bar(MODEL_NAMES, verbosity, color=colors)
axes[2].set_title('Verbosity (Output/Input Ratio)', fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 6. GPU Metrics Analysis 🖥️

In [ ]:
# GPU metrics overview
gpu_cols = ['gpu_utilization_pct', 'gpu_memory_used_mb', 'gpu_temperature_c', 'gpu_power_watts']
gpu_labels = ['Utilization (%)', 'Memory (MB)', 'Temperature (°C)', 'Power (W)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, label in zip(axes.flatten(), gpu_cols, gpu_labels):
    if col in df.columns:
        gpu_by_model = df.groupby('model_name')[col].mean().reindex(MODEL_NAMES)
        colors = [MODEL_COLORS[m] for m in MODEL_NAMES]
        ax.bar(MODEL_NAMES, gpu_by_model, color=colors)
        ax.set_title(f'Mean {label} by Model', fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.set_ylabel(label)
    else:
        ax.text(0.5, 0.5, f'{col} not available', ha='center', va='center', fontsize=12)
        ax.set_title(label, fontweight='bold')

plt.suptitle('GPU Metrics by Model', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# GPU metrics heatmap
available_gpu_cols = [c for c in gpu_cols if c in df.columns]
if available_gpu_cols:
    gpu_pivot = df.groupby('model_name')[available_gpu_cols].mean().reindex(MODEL_NAMES)
    gpu_pivot.columns = [c.replace('gpu_', '').replace('_', ' ').title() for c in gpu_pivot.columns]
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(gpu_pivot, annot=True, fmt='.1f', cmap='coolwarm', linewidths=0.5)
    plt.title('GPU Metrics by Model (Normalized)', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 7. Response Quality & Characteristics ✅

In [ ]:
# Success rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

success_rate = df.groupby('model_name')['ok'].mean().reindex(MODEL_NAMES) * 100
colors = [MODEL_COLORS[m] for m in MODEL_NAMES]
axes[0].bar(MODEL_NAMES, success_rate, color=colors)
axes[0].axhline(y=success_rate.mean(), color='red', linestyle='--', label=f'Mean: {success_rate.mean():.1f}%')
axes[0].set_title('Success Rate (ok=True) by Model', fontweight='bold')
axes[0].set_ylabel('Success Rate (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Error counts
error_counts = df[df['ok'] == False].groupby('model_name').size().reindex(MODEL_NAMES, fill_value=0)
axes[1].bar(MODEL_NAMES, error_counts, color=colors)
axes[1].set_title('Error Count by Model', fontweight='bold')
axes[1].set_ylabel('Errors')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Response length analysis
if 'response_text' in df.columns:
    df['response_length'] = df['response_text'].fillna('').str.len()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for model in MODEL_NAMES:
        data = df[df['model_name'] == model]['response_length']
        ax.hist(data, bins=50, alpha=0.5, label=model, color=MODEL_COLORS[model])
    ax.set_title('Response Length Distribution', fontweight='bold')
    ax.set_xlabel('Characters')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 8. Confidence Score Analysis 🎯

In [ ]:
# Confidence distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, model in zip(axes.flatten()[:5], MODEL_NAMES):
    data = df[df['model_name'] == model]['confidence_score'].dropna()
    ax.hist(data, bins=30, color=MODEL_COLORS[model], alpha=0.7, edgecolor='white')
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.3f}')
    ax.set_title(model, fontweight='bold')
    ax.set_xlabel('Confidence Score')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

axes.flatten()[5].axis('off')
plt.suptitle('Confidence Score Distribution by Model', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Calibration analysis
fig, ax = plt.subplots(figsize=(10, 8))

for model in MODEL_NAMES:
    model_df = df[(df['model_name'] == model) & df['confidence_score'].notna() & df['glider_score'].notna()]
    if len(model_df) > 0:
        # Bin by confidence
        model_df['conf_bin'] = pd.cut(model_df['confidence_score'], bins=10, labels=False)
        model_df['correct'] = model_df['glider_score'] >= 0.5
        calibration = model_df.groupby('conf_bin')['correct'].mean()
        conf_means = model_df.groupby('conf_bin')['confidence_score'].mean()
        ax.plot(conf_means, calibration, 'o-', label=model, color=MODEL_COLORS[model], linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
ax.set_xlabel('Mean Confidence')
ax.set_ylabel('Actual Accuracy')
ax.set_title('Calibration Plot (Reliability Diagram)', fontweight='bold')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 9. Performance by Task Type 📊

In [ ]:
# Accuracy heatmap by task
if 'glider_score' in df.columns:
    df['is_correct'] = df['glider_score'] >= 0.5
    acc_pivot = df.pivot_table(values='is_correct', index='router_task', columns='model_name', aggfunc='mean')
    
    plt.figure(figsize=(14, 20))
    sns.heatmap(acc_pivot, annot=True, fmt='.2f', cmap='RdYlGn', linewidths=0.5, 
                center=0.5, vmin=0, vmax=1)
    plt.title('Accuracy by Task × Model', fontsize=16, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('Task')
    plt.tight_layout()
    plt.show()

In [ ]:
# Best model per task
if 'glider_score' in df.columns:
    best_model = acc_pivot.idxmax(axis=1)
    best_model_counts = best_model.value_counts()
    
    plt.figure(figsize=(10, 6))
    colors = [MODEL_COLORS.get(m, '#888888') for m in best_model_counts.index]
    plt.bar(best_model_counts.index, best_model_counts.values, color=colors)
    plt.title('Number of Tasks Where Each Model is Best', fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('# Tasks')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

---
## 10. Performance by Ground Truth Type 🎲

In [ ]:
# Accuracy by GT type
if 'ground_truth_type' in df.columns and 'is_correct' in df.columns:
    gt_pivot = df.pivot_table(values='is_correct', index='ground_truth_type', columns='model_name', aggfunc='mean')
    
    fig, ax = plt.subplots(figsize=(12, 6))
    gt_pivot.plot(kind='bar', ax=ax, color=[MODEL_COLORS.get(m, '#888888') for m in gt_pivot.columns])
    ax.set_title('Accuracy by Ground Truth Type × Model', fontweight='bold')
    ax.set_xlabel('Ground Truth Type')
    ax.set_ylabel('Accuracy')
    ax.legend(title='Model', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

---
## 11. Performance by Data Split 🔀

In [ ]:
# Performance by split
if 'split' in df.columns and 'is_correct' in df.columns:
    split_pivot = df.pivot_table(values='is_correct', index='split', columns='model_name', aggfunc='mean')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    split_pivot.plot(kind='bar', ax=ax, color=[MODEL_COLORS.get(m, '#888888') for m in split_pivot.columns])
    ax.set_title('Accuracy by Data Split (Train/Val/Test)', fontweight='bold')
    ax.set_xlabel('Split')
    ax.set_ylabel('Accuracy')
    ax.legend(title='Model', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

---
## 12. Cross-Model Correlation 🔗

In [ ]:
# Agreement matrix
if 'is_correct' in df.columns:
    # Pivot to get correctness per sample per model
    correct_pivot = df.pivot_table(values='is_correct', index='sample_id', columns='model_name', aggfunc='first')
    
    # Calculate correlation
    corr_matrix = correct_pivot.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, linewidths=0.5)
    plt.title('Model Agreement Correlation (is_correct)', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Agreement counts
if 'is_correct' in df.columns:
    correct_pivot['num_correct'] = correct_pivot.sum(axis=1)
    agreement_dist = correct_pivot['num_correct'].value_counts().sort_index()
    
    plt.figure(figsize=(10, 6))
    plt.bar(agreement_dist.index, agreement_dist.values, color='steelblue')
    plt.xlabel('Number of Models Correct')
    plt.ylabel('Number of Samples')
    plt.title('Sample Difficulty Distribution', fontweight='bold')
    plt.xticks(range(6))
    
    # Add labels
    easy = agreement_dist.get(5, 0)
    hard = agreement_dist.get(0, 0)
    plt.annotate(f'Easy: {easy}', xy=(5, easy), ha='center', va='bottom', fontsize=10)
    plt.annotate(f'Hard: {hard}', xy=(0, hard), ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()

---
## 13. Cost-Performance Tradeoffs ⚖️

In [ ]:
# Pareto frontier: Cost vs Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Cost vs Accuracy
model_stats = []
for model in MODEL_NAMES:
    model_df = df[df['model_name'] == model]
    acc = model_df['is_correct'].mean() if 'is_correct' in model_df.columns else model_df['ok'].mean()
    cost = model_df['estimated_cost'].mean() * 1000  # per 1000 samples
    latency = model_df['latency_ms'].mean()
    model_stats.append({'model': model, 'accuracy': acc, 'cost': cost, 'latency': latency})

stats_df = pd.DataFrame(model_stats)

for _, row in stats_df.iterrows():
    axes[0].scatter(row['cost'], row['accuracy'], s=200, label=row['model'], 
                    color=MODEL_COLORS[row['model']], edgecolors='black')
axes[0].set_xlabel('Cost per 1000 Samples ($)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Cost vs Accuracy', fontweight='bold')
axes[0].legend()

# Latency vs Accuracy
for _, row in stats_df.iterrows():
    axes[1].scatter(row['latency'], row['accuracy'], s=200, label=row['model'],
                    color=MODEL_COLORS[row['model']], edgecolors='black')
axes[1].set_xlabel('Mean Latency (ms)')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Latency vs Accuracy', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for multi-objective comparison
from math import pi

# Normalize metrics (higher is better)
stats_df['norm_acc'] = stats_df['accuracy']
stats_df['norm_speed'] = 1 - (stats_df['latency'] - stats_df['latency'].min()) / (stats_df['latency'].max() - stats_df['latency'].min())
stats_df['norm_cost'] = 1 - (stats_df['cost'] - stats_df['cost'].min()) / (stats_df['cost'].max() - stats_df['cost'].min())

categories = ['Accuracy', 'Speed', 'Cost Efficiency']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for _, row in stats_df.iterrows():
    values = [row['norm_acc'], row['norm_speed'], row['norm_cost']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['model'], color=MODEL_COLORS[row['model']])
    ax.fill(angles, values, alpha=0.1, color=MODEL_COLORS[row['model']])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_title('Multi-Objective Model Comparison', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.show()

---
## 14. Final Summary & Recommendations 📝

In [ ]:
# Final rankings
rankings = stats_df.copy()
rankings['accuracy_rank'] = rankings['accuracy'].rank(ascending=False).astype(int)
rankings['latency_rank'] = rankings['latency'].rank().astype(int)
rankings['cost_rank'] = rankings['cost'].rank().astype(int)
rankings['overall_rank'] = (rankings['accuracy_rank'] + rankings['latency_rank'] + rankings['cost_rank']).rank().astype(int)

rankings_display = rankings[['model', 'accuracy', 'latency', 'cost', 'accuracy_rank', 'latency_rank', 'cost_rank', 'overall_rank']]
rankings_display = rankings_display.sort_values('overall_rank')
rankings_display.columns = ['Model', 'Accuracy', 'Latency (ms)', 'Cost ($/1k)', 'Acc Rank', 'Lat Rank', 'Cost Rank', 'Overall']

print("\n📊 Model Rankings:")
display(rankings_display)

In [ ]:
# Key insights summary
print("\n" + "="*60)
print("🔑 KEY INSIGHTS")
print("="*60)

# Best accuracy
best_acc_model = stats_df.loc[stats_df['accuracy'].idxmax(), 'model']
best_acc = stats_df['accuracy'].max()
print(f"\n📈 Best Accuracy: {best_acc_model} ({best_acc:.1%})")

# Fastest
fastest_model = stats_df.loc[stats_df['latency'].idxmin(), 'model']
fastest_lat = stats_df['latency'].min()
print(f"⚡ Fastest: {fastest_model} ({fastest_lat:.0f}ms)")

# Cheapest
cheapest_model = stats_df.loc[stats_df['cost'].idxmin(), 'model']
cheapest_cost = stats_df['cost'].min()
print(f"💰 Most Cost-Efficient: {cheapest_model} (${cheapest_cost:.2f}/1k)")

# Total benchmark stats
print(f"\n📊 Total Benchmark Statistics:")
print(f"   - Samples: {len(df_samples):,}")
print(f"   - Responses: {len(df_responses):,}")
print(f"   - Evaluations: {len(df_evaluations):,}")
print(f"   - Total Cost: ${df['estimated_cost'].sum():.2f}")
print(f"   - Tasks: {df['router_task'].nunique()}")
print("="*60)